# EXP2: Method Comparison

Compare six velocity-estimation methods on three synthetic flow datasets using velocity and gradient-based metrics.


In [ ]:
# The basic funcs
import json
import hashlib
import importlib
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "generator").is_dir() and (path / "methods").is_dir():
            return path
    raise RuntimeError("Cannot find project root. Please run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.flow import FlowManager
from generator.images import config_events
from generator.loader import EventManager

import methods.method as method_api
importlib.reload(method_api)
from methods.method import inr_vg,inr_est,of_vg,of_est, ilk_vg, ilk_est, affineGS, constGS, compute_flow_quantities, EVENT_WINDOW_MS, TEMPORAL_NUM_BINS

from utils.metrics import PIVmetric
from utils.visualizer import plot_flow_field,plot_event_time2d, create_colorbar
from utils.painter import plot_vortex_line_panel

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)              
torch.cuda.manual_seed(seed)         
torch.cuda.manual_seed_all(seed)    

# ====================== PyTorch ============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

SINGLE_FLOW_OUTPUT_DIR = PROJECT_ROOT / "output" / "EXP2" / "on_single_flow"
SINGLE_FLOW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STARTED_AT = datetime.now().astimezone()
RUN_STARTED_TS = RUN_STARTED_AT.timestamp()
RUN_ID = RUN_STARTED_AT.isoformat(timespec="seconds")

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    temporary.replace(path)

SOURCE_FILES = [
    PROJECT_ROOT / "methods/method.py",
    PROJECT_ROOT / "methods/affinegs/gaussian_flow/config.py",
    PROJECT_ROOT / "methods/affinegs/gaussian_flow/training.py",
    PROJECT_ROOT / "methods/affinegs/gaussian_flow/model.py",
    PROJECT_ROOT / "generator/images.py",
    PROJECT_ROOT / "generator/flow.py",
    PROJECT_ROOT / "utils/metrics.py",
]
SOURCE_SHA256 = {str(path.relative_to(PROJECT_ROOT)): sha256_file(path) for path in SOURCE_FILES}
print("Run ID:", RUN_ID)

# Test Data

In [ ]:
FLOW_DATA_PATHS = [
    PROJECT_ROOT / "data" / "datasets" / "JHTDB_isotropic1024_hd_00001_flow.npz",
    PROJECT_ROOT / "data" / "datasets" / "DNS_turbulence_00001_flow.npz",
    PROJECT_ROOT / "data" / "datasets" / "SQG_00001_flow.npz",
]
flow0, flow1, flow2 = [FlowManager(mode="data", data_path=str(path)) for path in FLOW_DATA_PATHS]
flow_names = ["JHTDB-isotropic", "DNS-turbulence", "SQG"]
flow_file_stems = ["JHTDB_isotropic", "DNS_turbulence", "SQG"]
FLOW_INPUT_SHA256 = {name: sha256_file(path) for name, path in zip(flow_names, FLOW_DATA_PATHS)}


In [ ]:
def flow_metrics(div_pred, div_true, omega_pred, omega_true):
    abs_div = np.abs(div_pred)
    return {
        "mean_abs_div" : float(abs_div.mean()),
        "rmse_div"     : float(np.sqrt(np.mean((div_pred - div_true) ** 2))),
        "rmse_omega"   : float(np.sqrt(np.mean((omega_pred - omega_true) ** 2))),
    }

def build_wide_metrics_table(metrics_dict_list, flow_name_list=flow_names, method_order=None, round_digits=4):

    # 行顺序：六种方法
    if method_order is None:
        method_order = list(metrics_dict_list[0].keys())

    metrics_order = ["rmse", "aee", "RMSE_div", "RMSE_ω"]

    # 构建二维数据
    data = {}
    for flow_name, metrics_dict in zip(flow_name_list, metrics_dict_list):
        for metric_name in metrics_order:
            col_key = (flow_name, metric_name)
            data[col_key] = [
                float(metrics_dict.get(method, {}).get(metric_name, float("nan")))
                for method in method_order
            ]

    df = pd.DataFrame(data, index=method_order)
    df.index.name = "method"
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    df = df.round(round_digits)

    return df



# Methods

In [ ]:
estimators = [ilk_vg, of_vg, inr_est, inr_vg, constGS, affineGS]
method_names = ["ILK_VG", "OF_VG", "INR_EST", "INR_VG", "ConstGS", "AffineGS"]
single_flow_run_records = {}

def test(flow_index, coord_idx, H=256, W=256, error_map=True, plot_vor=True):

    flow_func = globals()[f"flow{flow_index}"].evaluate
    cfg_evts = config_events()
    cfg_evts.time_length = EVENT_WINDOW_MS  # 3 ms for all methods
    cfg_evts.d = 0.5
    cfg_evts.density = 0.05

    evts, (u, v) = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func).load()
    evts = evts[(evts[:, 2] > 0) & (evts[:, 2] < EVENT_WINDOW_MS)]
    v_h_gt = u.squeeze().cpu().numpy()
    v_w_gt = v.squeeze().cpu().numpy()
    div_gt, vor_gt = compute_flow_quantities(v_h_gt, v_w_gt)

    vor_list = [vor_gt]   # 第0通道 = GT
    names = ["ground truth"]

    # 新增：用于保存各方法指标
    metrics_dict = {}

    for i, m in enumerate(estimators):
        
        v_h, v_w, div, vor = m(evts, H=H, W=W)
        rmse, aee = PIVmetric(v_h, v_w, v_h_gt, v_w_gt, show=False)[:2]
        # print(f"results：rmse: {rmse:.4f}、aee: {aee:.4f}")

        # Use the same finite-difference operator for every method, including ConstGS/AffineGS.
        div_eval, vor_eval = compute_flow_quantities(v_h, v_w)
        metric_vals = flow_metrics(
            div_eval.detach().numpy(),
            div_gt.detach().numpy(),
            vor_eval.detach().numpy(),
            vor_gt.detach().numpy(),
        )
        # print(f"mean|div| : {metric_vals['mean_abs_div']:.4f}")
        # print(f"RMSE_div  : {metric_vals['rmse_div']:.4f}")
        # print(f"RMSE_ω    : {metric_vals['rmse_omega']:.4f}")

        # 新增：保存到字典
        metrics_dict[method_names[i]] = {
            "rmse": float(rmse),
            "aee": float(aee),
            "mean|div|": float(metric_vals["mean_abs_div"]),
            "RMSE_div": float(metric_vals["rmse_div"]),
            "RMSE_ω": float(metric_vals["rmse_omega"]),
        }

        names.append(method_names[i])
        vor_list.append(vor_eval)
        
        if error_map:
            path = SINGLE_FLOW_OUTPUT_DIR / f"{flow_file_stems[flow_index]}_{method_names[i]}.svg"
            path.parent.mkdir(parents=True, exist_ok=True)

            plot_flow_field(v_h, v_w, v_h_gt, v_w_gt, output_file=str(path), magFlag=True, slFlag=False, colorbar_flag=False, step=1, quiver_step=32, 
                            v_min=0, v_max=0.5, scale=None, figsize=(2, 2), cbar_ticks=None, cbar_tick_size=12)

    vor_stack = torch.stack(vor_list, axis=0)   # [C, H, W]

    # Every completed case is persisted immediately; notebook autosave is not required.
    flow_name = flow_names[flow_index]
    record = {
        "schema_version": 1,
        "run_id": RUN_ID,
        "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "flow_name": flow_name,
        "flow_input": str(FLOW_DATA_PATHS[flow_index]),
        "flow_input_sha256": FLOW_INPUT_SHA256[flow_name],
        "source_sha256": SOURCE_SHA256,
        "config": {
            "seed": seed, "H": H, "W": W,
            "event_window_ms": float(EVENT_WINDOW_MS),
            "temporal_num_bins": int(TEMPORAL_NUM_BINS),
            "particle_spacing": float(cfg_evts.d),
            "particle_density": float(cfg_evts.density),
            "event_count": int(len(evts)),
        },
        "methods": method_names,
        "metrics": metrics_dict,
    }
    metrics_path = SINGLE_FLOW_OUTPUT_DIR / f"{flow_file_stems[flow_index]}-metrics.json"
    write_json_atomic(metrics_path, record)
    vorticity_path = SINGLE_FLOW_OUTPUT_DIR / f"{flow_file_stems[flow_index]}-vorticity.npz"
    temporary_vorticity = vorticity_path.with_suffix(".npz.tmp")
    with open(temporary_vorticity, "wb") as f:
        np.savez_compressed(
            f, omega=vor_stack.detach().cpu().numpy(),
            method_names=np.asarray(["ground truth", *method_names]),
            coord_idx=int(coord_idx), run_id=RUN_ID,
        )
    temporary_vorticity.replace(vorticity_path)
    record["artifacts"] = {
        "metrics": str(metrics_path),
        "vorticity": str(vorticity_path),
    }
    single_flow_run_records[flow_name] = record
    print(f"Saved current {flow_name} metrics and vorticity (run_id={RUN_ID})")

    # 新增：返回指标字典
    return vor_stack, metrics_dict



In [ ]:
vor_stack0, metrics_dict0= test(flow_index = 0, coord_idx=128)
torch.cuda.empty_cache()

In [ ]:
vorticity_figure_paths = {
    name: SINGLE_FLOW_OUTPUT_DIR / f"{stem}-vorticity.pdf"
    for name, stem in zip(flow_names, flow_file_stems)
}
plot_vortex_line_panel(omega=vor_stack0, coord_idx=128, yticks=[-0.8, -0.4, 0, 0.4, 0.8], cbar_ticks=[-2, -1, 0, 1, 2], save_path=str(vorticity_figure_paths[flow_names[0]]))
legacy_fig10_path = PROJECT_ROOT / "output" / "EXP2" / "fig10.pdf"
shutil.copy2(vorticity_figure_paths[flow_names[0]], legacy_fig10_path)
print(f"Saved: {vorticity_figure_paths[flow_names[0]]} (compatibility copy: {legacy_fig10_path})")

In [ ]:
vor_stack1, metrics_dict1 = test(flow_index = 1, coord_idx=128)
torch.cuda.empty_cache()

In [ ]:
plot_vortex_line_panel(omega=vor_stack1, coord_idx=128, yticks=[-0.5, -0.25, 0, 0.25, 0.5], cbar_ticks=[-1, -0.5, 0, 0.5, 1], save_path=str(vorticity_figure_paths[flow_names[1]]))
print(f"Saved: {vorticity_figure_paths[flow_names[1]]}")

In [ ]:
vor_stack2, metrics_dict2 = test(flow_index = 2, coord_idx=128)
torch.cuda.empty_cache()

In [ ]:
plot_vortex_line_panel(omega=vor_stack2, coord_idx=128, yticks=[-0.5, -0.25, 0, 0.25, 0.5], cbar_ticks=[-1, -0.5, 0, 0.5, 1], save_path=str(vorticity_figure_paths[flow_names[2]]))
print(f"Saved: {vorticity_figure_paths[flow_names[2]]}")

In [ ]:
current_source_sha256 = {str(path.relative_to(PROJECT_ROOT)): sha256_file(path) for path in SOURCE_FILES}
current_flow_input_sha256 = {name: sha256_file(path) for name, path in zip(flow_names, FLOW_DATA_PATHS)}
if current_source_sha256 != SOURCE_SHA256 or current_flow_input_sha256 != FLOW_INPUT_SHA256:
    raise RuntimeError("Source code or input files changed during the run; restart the kernel and run again.")

expected_flows = set(flow_names)
completed_flows = set(single_flow_run_records)
if completed_flows != expected_flows:
    raise RuntimeError(f"Incomplete current run: expected {sorted(expected_flows)}, got {sorted(completed_flows)}")
record_run_ids = {record["run_id"] for record in single_flow_run_records.values()}
if record_run_ids != {RUN_ID}:
    raise RuntimeError(f"Mixed run IDs detected: current={RUN_ID}, records={sorted(record_run_ids)}")

metrics_dict_list = [metrics_dict0, metrics_dict1, metrics_dict2]
wide_metrics_table = build_wide_metrics_table(
    metrics_dict_list=metrics_dict_list, flow_name_list=flow_names,
    method_order=method_names, round_digits=4,
)

table1_data = {}
table2_data = {}
for flow_name, metrics in zip(flow_names, metrics_dict_list):
    table1_data[(flow_name, "RMSE")] = [metrics[m]["rmse"] for m in method_names]
    table1_data[(flow_name, "AEE")] = [metrics[m]["aee"] for m in method_names]
    table2_data[(flow_name, "RMSE_div")] = [metrics[m]["RMSE_div"] for m in method_names]
    table2_data[(flow_name, "RMSE_omega")] = [metrics[m]["RMSE_ω"] for m in method_names]
table1 = pd.DataFrame(table1_data, index=method_names).round(4)
table2 = pd.DataFrame(table2_data, index=method_names).round(4)
table1.index.name = table2.index.name = "method"
table1.columns = pd.MultiIndex.from_tuples(table1.columns)
table2.columns = pd.MultiIndex.from_tuples(table2.columns)

table1_path = SINGLE_FLOW_OUTPUT_DIR / "table1-velocity-metrics.csv"
table2_path = SINGLE_FLOW_OUTPUT_DIR / "table2-gradient-metrics.csv"
for table, path in ((table1, table1_path), (table2, table2_path)):
    temporary = path.with_suffix(path.suffix + ".tmp")
    table.to_csv(temporary)
    temporary.replace(path)

results_path = SINGLE_FLOW_OUTPUT_DIR / "single-flow-results.json"
write_json_atomic(results_path, {
    "schema_version": 1, "run_id": RUN_ID,
    "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "source_sha256": SOURCE_SHA256,
    "flow_input_sha256": FLOW_INPUT_SHA256,
    "records": single_flow_run_records,
})

artifact_paths = [results_path, table1_path, table2_path, legacy_fig10_path]
for flow_index, flow_name in enumerate(flow_names):
    stem = flow_file_stems[flow_index]
    artifact_paths.extend([
        SINGLE_FLOW_OUTPUT_DIR / f"{stem}-metrics.json",
        SINGLE_FLOW_OUTPUT_DIR / f"{stem}-vorticity.npz",
        vorticity_figure_paths[flow_name],
    ])
    artifact_paths.extend(
        SINGLE_FLOW_OUTPUT_DIR / f"{stem}_{method}.svg" for method in method_names
    )
missing_artifacts = [str(path) for path in artifact_paths if not path.is_file()]
if missing_artifacts:
    raise RuntimeError("Missing artifacts from current run:\n" + "\n".join(missing_artifacts))
stale_artifacts = [str(path) for path in artifact_paths if path.stat().st_mtime + 1 < RUN_STARTED_TS]
if stale_artifacts:
    raise RuntimeError("Artifacts were not regenerated in the current run:\n" + "\n".join(stale_artifacts))
manifest_path = SINGLE_FLOW_OUTPUT_DIR / "run-manifest.json"
write_json_atomic(manifest_path, {
    "schema_version": 1, "run_id": RUN_ID,
    "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "config": {
        "seed": seed, "event_window_ms": float(EVENT_WINDOW_MS),
        "temporal_num_bins": int(TEMPORAL_NUM_BINS),
        "resolution": [256, 256], "flow_names": flow_names,
        "method_names": method_names,
    },
    "source_sha256": SOURCE_SHA256,
    "flow_input_sha256": FLOW_INPUT_SHA256,
    "artifacts": {
        str(path.relative_to(PROJECT_ROOT)): {"sha256": sha256_file(path), "size": path.stat().st_size}
        for path in artifact_paths
    },
})
print(f"Saved current tables, results, and manifest for run_id={RUN_ID}")
wide_metrics_table

In [ ]:
single_flow_output_dir = PROJECT_ROOT / "output" / "EXP2" / "on_single_flow"
single_flow_output_dir.mkdir(parents=True, exist_ok=True)

for flow_index in range(len(flow_names)):
    colorbar_path = single_flow_output_dir / f"{flow_file_stems[flow_index]}.svg"
    create_colorbar(
        0,
        0.5,
        figsize=(1.6, 4.8),
        fontsize=24,
        orientation="vertical",
        output_file=str(colorbar_path),
        label="EPE [px/ms]",
    )

## Datasets

In [ ]:
def save_results_to_json(results, save_path, indent=1):
    def convert_numpy_types(obj):
        if isinstance(obj, (np.integer, np.floating)): 
            return obj.item()
        elif isinstance(obj, np.ndarray): 
            return obj.tolist()
        return obj
    
    try:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=indent, default=convert_numpy_types)
        print(f"✅ Results saved to: {save_path}")
    except Exception as e:
        print(f"❌ Save failed: {e}")

In [ ]:
DATA_ROOT = "/media/newdisk/datasets_piv/piv_cai/train"
OUTPUT_DIR = PROJECT_ROOT / "output" / "EXP2" / "on_datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# name: result JSON basename kept compatible with existing files.
# display_name: label used in figures/logs.
dataset_specs = [
    {"name": "JHTDB-isotropic", "display_name": "JHTDB-isotropic", "file_prefix": "JHTDB_isotropic1024_hd"},
    {"name": "DNS-turbulence", "display_name": "DNS-turbulence", "file_prefix": "DNS_turbulence"},
    {"name": "SQG", "display_name": "SQG", "file_prefix": "SQG"},
]
datasets_name = [spec["display_name"] for spec in dataset_specs]
selected_data_indices = [2]

if selected_data_indices:
    selected_dataset_specs = [dataset_specs[i] for i in selected_data_indices]
else:
    selected_dataset_specs = dataset_specs

print("Datasets to run:", [spec["display_name"] for spec in selected_dataset_specs])

estimators = [ilk_vg, of_vg, inr_est, inr_vg, constGS, affineGS]
method_names = ["ILK_VG", "OF_VG", "INR_EST", "INR_VG", "ConstGS", "AffineGS"]

max_valid_files = 1000
file_idx_range = range(1, 1201)
resume_existing = False  # Set True to continue from existing JSON files after an interrupted run.

all_dataset_results = {}

for data_index, dataset_spec in enumerate(tqdm(selected_dataset_specs, desc="Datasets")):
    dataset_name = dataset_spec["name"]
    dataset_display_name = dataset_spec["display_name"]
    file_prefix = dataset_spec["file_prefix"]
    result_path = OUTPUT_DIR / f"{dataset_name}({max_valid_files}).json"

    # Optional resume for interrupted long runs. Default is a fresh run for all 3 datasets.
    if resume_existing and os.path.exists(result_path):
        with open(result_path, "r", encoding="utf-8") as f:
            final_results = json.load(f)
        print(f"[{dataset_display_name}] resume from {len(final_results)} saved files: {result_path}")
    else:
        final_results = {}

    for file_idx in tqdm(file_idx_range, desc=f"{dataset_display_name}", leave=False):
        if len(final_results) >= max_valid_files:
            break

        file_key = f"{file_idx:05d}"
        if file_key in final_results and all(name in final_results[file_key] for name in method_names):
            continue

        file_name = f"{file_prefix}_{file_idx:05d}.npz"
        file_path = os.path.join(DATA_ROOT, file_name)
        if not os.path.exists(file_path):
            continue

        flow = FlowManager(mode="data", data_path=file_path)
        flow_func = flow.evaluate

        cfg_evts = config_events()
        cfg_evts.time_length = EVENT_WINDOW_MS  # 3 ms for all methods
        cfg_evts.d = 0.5
        cfg_evts.density = 0.05

        evts, (u, v) = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func).load()
        evts = evts[(evts[:, 2] > 0) & (evts[:, 2] < EVENT_WINDOW_MS)]
        v_h_gt = u.squeeze().cpu().numpy()
        v_w_gt = v.squeeze().cpu().numpy()
        div_gt, vor_gt = compute_flow_quantities(v_h_gt, v_w_gt)

        file_results = {}
        for method, name in zip(estimators, method_names):
            v_h, v_w, div, vor = method(evts, H=256, W=256)
            rmse, aee = PIVmetric(v_h, v_w, v_h_gt, v_w_gt, show=False)[:2]
            div_eval, vor_eval = compute_flow_quantities(v_h, v_w)
            metric_vals = flow_metrics(
                div_eval.detach().numpy(),
                div_gt.detach().numpy(),
                vor_eval.detach().numpy(),
                vor_gt.detach().numpy(),
            )
            file_results[name] = {
                "rmse": round(float(rmse), 4),
                "aee": round(float(aee), 4),
                "mean|div|": round(float(metric_vals["mean_abs_div"]), 4),
                "RMSE_div": round(float(metric_vals["rmse_div"]), 4),
                "RMSE_ω": round(float(metric_vals["rmse_omega"]), 4),
            }

        final_results[file_key] = file_results

        if len(final_results) % 200 == 0:
            checkpoint_path = OUTPUT_DIR / f"{dataset_name}({len(final_results)}).json"
            save_results_to_json(final_results, checkpoint_path, indent=1)
            save_results_to_json(final_results, result_path, indent=1)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if final_results:
        save_results_to_json(final_results, result_path, indent=1)
    else:
        print(f"[{dataset_display_name}] no matching files found for prefix '{file_prefix}' in {DATA_ROOT}; nothing saved.")

    all_dataset_results[dataset_display_name] = final_results
    print(f"[{dataset_display_name}] finished {len(final_results)} files -> {result_path}")

In [ ]:
from utils.painter import plot_boxplot

dataset_files = ["JHTDB-isotropic", "DNS-turbulence", "SQG"]
datasets_name = ["JHTDB-isotropic", "DNS-turbulence", "SQG"]

def plot(data_index, dataset_files, datasets_name, use_Chinese = False, save=False):
    dataset_name = dataset_files[data_index]
    file_path = PROJECT_ROOT / "output" / "EXP2" / "on_datasets" / f"{dataset_name}(1000).json"

    if save is True:
        rmse_path = str(PROJECT_ROOT / "output" / "EXP2" / "on_datasets" / f"{dataset_name}-rmse.pdf")
        aee_path = str(PROJECT_ROOT / "output" / "EXP2" / "on_datasets" / f"{dataset_name}-aee.pdf")
    else:
        rmse_path = None
        aee_path = None

    if use_Chinese is True:
        size = [18, 10]
    else:
        size = [12, 10]
        
    plot_boxplot(str(file_path), metric='rmse', save_path=rmse_path, 
        ylabel_en="RMSE $[px/ms]$", ylabel_zh="RMSE $[px/ms]$",
        xlabel_en='Methods', xlabel_zh='六种方法',
        title_en=None, title_zh=None,
        box_width=0.5, y_lim=(0, 0.8), 
        label_size=size[0], tick_size=size[1], 
        grid=True, use_chinese=use_Chinese, dataset_name=datasets_name[data_index]
    )
    
    plot_boxplot(str(file_path), metric='aee', save_path=aee_path, 
        ylabel_en="AEE $[px/ms]$", ylabel_zh="AEE $[px/ms]$",
        xlabel_en='Methods', xlabel_zh='六种方法',
        title_en=None, title_zh=None,
        box_width=0.5, y_lim=(0,0.8), 
        label_size=size[0], tick_size=size[1], 
        grid=True, use_chinese=use_Chinese, dataset_name=datasets_name[data_index]
    )

for data_index in range(3):
    print(f"dataset:{datasets_name[data_index]}")
    plot(data_index=data_index, dataset_files=dataset_files, datasets_name=datasets_name, use_Chinese = False, save=True)
